# Findrr ML-001: Title Expander - training and export (v2.0.0)

Implements `docs/ml/models/title-expander.md`: fine-tune `google/flan-t5-small` to
generate alternative job titles from a title, skills and a seniority level.

**Data:** O*NET 31.0 (public, CC BY 4.0): reported titles, job titles, software skills.
No real user data. Splits keep every occupation in exactly one of train/val/test.

**Run on Kaggle:** Settings -> Accelerator **GPU T4**, **Internet on** -> Run All.
Then download `title_expander_bundle.tar.gz` and extract it into
`models/title-expander/v2.0.0/` in your Findrr folder.

The notebook stops before export if the model fails the approved acceptance
criteria (ROUGE-L >= 0.75, valid title ratio >= 0.85, repetition < 0.05).

In [ ]:
# Environment report (useful if an install or training step fails).
import platform, subprocess, sys
print('Python', sys.version)
print(platform.platform())
print(subprocess.run([sys.executable, '-m', 'pip', 'list'], capture_output=True, text=True).stdout
      .count('\n'), 'packages preinstalled')
for pkg in ('torch', 'transformers', 'tokenizers', 'numpy', 'onnxruntime'):
    try:
        print(pkg, __import__(pkg).__version__)
    except Exception as err:
        print(pkg, 'not available:', type(err).__name__)

In [ ]:
# Pure-Python libraries whose APIs this notebook uses are pinned exactly.
# Compiled packages (torch, tokenizers, safetensors, numpy) come from the Kaggle
# image: pinning old releases forces source builds on newer Python versions.
# sentencepiece is not needed: flan-t5 loads its fast tokenizer from tokenizer.json.
!pip install -q --prefer-binary transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 \
    rouge-score==0.1.2 sacrebleu==2.4.3

In [ ]:
# O*NET database (CC BY 4.0, U.S. Department of Labor / ETA). Public occupational data only;
# no real user data is used anywhere in this notebook.
ONET_VERSION = '31_0'
ONET_URL = f'https://www.onetcenter.org/dl_files/database/db_{ONET_VERSION}_text.zip'
# SHA-256 of db_31_0_text.zip as downloaded on 2026-11-26. Change both values together.
ONET_SHA256 = '6883548adf5fde64cf6f801b35d15519c9225f2732c3cab0e281c652d16b23a9'
# Optional offline copy: attach the extracted folder as a Kaggle dataset at this path.
ONET_OFFLINE_DIR = '/kaggle/input/onet-db'
SEED = 42

In [ ]:
import hashlib, io, os, zipfile, urllib.request
from pathlib import Path

def obtain_onet_dir() -> Path:
    """Return a directory with the O*NET text files; verify the archive checksum."""
    offline = Path(ONET_OFFLINE_DIR)
    if offline.exists():
        hits = list(offline.rglob('Occupation Data.txt'))
        if hits:
            print(f'Using attached O*NET dataset at {hits[0].parent}')
            return hits[0].parent
    print(f'Downloading {ONET_URL} ...')
    with urllib.request.urlopen(ONET_URL, timeout=120) as resp:
        blob = resp.read()
    digest = hashlib.sha256(blob).hexdigest()
    if digest != ONET_SHA256:
        raise RuntimeError(
            f'O*NET archive checksum mismatch: got {digest}, expected {ONET_SHA256}. '
            'Refusing to train on unverified data.'
        )
    target = Path('/kaggle/working/onet')
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        zf.extractall(target)
    return next(target.rglob('Occupation Data.txt')).parent

ONET_DIR = obtain_onet_dir()

In [ ]:
import csv, re, random
from collections import defaultdict
from pathlib import Path

def _find(root: Path, *names: str) -> Path:
    """Return the first existing file among names (O*NET renamed files across releases)."""
    for name in names:
        p = Path(root) / name
        if p.exists():
            return p
    raise FileNotFoundError(f'None of {names} found in {root}')

def _rows(path: Path):
    with open(path, encoding='utf-8', newline='') as fh:
        yield from csv.DictReader(fh, delimiter='\t')

_PAREN = re.compile(r'\s*\([^)]*\)')

def clean_title(title: str, short: str = '') -> str:
    """Prefer the O*NET short title; drop parenthetical expansions; collapse whitespace."""
    base = short if short and short.strip().lower() not in ('n/a', '') else title
    base = _PAREN.sub('', base)
    return ' '.join(base.replace('/', ' / ').split()).strip(' ,;')

def norm(title: str) -> str:
    return ' '.join(re.sub(r'[^\w\s]', ' ', title.lower()).split())

def load_onet(root) -> dict:
    root = Path(root)
    occupations = {r['O*NET-SOC Code']: r['Title'] for r in _rows(_find(root, 'Occupation Data.txt'))}
    reported = defaultdict(list)
    for r in _rows(_find(root, 'Sample of Reported Titles.txt')):
        t = clean_title(r['Reported Job Title'])
        if t and t not in reported[r['O*NET-SOC Code']]:
            reported[r['O*NET-SOC Code']].append(t)
    job_titles = defaultdict(list)
    for r in _rows(_find(root, 'Job Titles.txt', 'Alternate Titles.txt')):
        title_col = 'Job Title' if 'Job Title' in r else 'Alternate Title'
        t = clean_title(r[title_col], r.get('Short Title', ''))
        if t and t not in job_titles[r['O*NET-SOC Code']]:
            job_titles[r['O*NET-SOC Code']].append(t)
    skills = defaultdict(list)
    for r in _rows(_find(root, 'Software Skills.txt', 'Technology Skills.txt')):
        ex = r.get('Workplace Example') or r.get('Example') or ''
        if ex:
            skills[r['O*NET-SOC Code']].append((ex.strip(), r.get('Hot Technology', 'N') == 'Y'))
    related = defaultdict(list)
    for r in _rows(_find(root, 'Related Occupations.txt')):
        related[r['O*NET-SOC Code']].append((r['Related O*NET-SOC Code'], r['Relatedness Tier']))
    return {
        'occupations': occupations,
        'reported': dict(reported),
        'job_titles': dict(job_titles),
        'skills': dict(skills),
        'related': dict(related),
    }

def split_by_occupation(socs, seed: int = 42, fractions=(0.8, 0.1, 0.1)) -> dict:
    """Assign whole occupations to train/val/test, stratified by SOC major group.

    Every title of an occupation lands in exactly one split, so evaluation titles are
    never seen in training (spec: disjoint titles, stratified by occupational domain).
    """
    by_group = defaultdict(list)
    for soc in sorted(set(socs)):
        by_group[soc[:2]].append(soc)
    rng = random.Random(seed)
    assignment = {}
    for group in sorted(by_group):
        members = by_group[group]
        rng.shuffle(members)
        n = len(members)
        n_val = max(1, round(n * fractions[1])) if n >= 3 else 0
        n_test = max(1, round(n * fractions[2])) if n >= 3 else 0
        for i, soc in enumerate(members):
            if i < n_test:
                assignment[soc] = 'test'
            elif i < n_test + n_val:
                assignment[soc] = 'val'
            else:
                assignment[soc] = 'train'
    return assignment

In [ ]:
# Dataset construction parameters (dataset shape, not acceptance thresholds).
LEVELS = ['ANY', 'ENTRY', 'MID', 'SENIOR', 'LEAD']  # == findrr SeniorityLevel + ANY
LEVEL_PATTERNS = {
    'ENTRY': r'\b(junior|jr|entry|associate|trainee|apprentice|intern|graduate)\b',
    'SENIOR': r'\b(senior|sr)\b',
    'LEAD': r'\b(lead|principal|chief|head)\b',
}
ANY_LEVEL = re.compile('|'.join(LEVEL_PATTERNS.values()), re.IGNORECASE)
MAX_ANCHORS_PER_OCCUPATION = 6
MAX_TARGET_TITLES = 5
MIN_TARGET_TITLES = 3
MAX_SKILLS = 5

def level_pool(level, neutral, all_titles):
    if level in LEVEL_PATTERNS:
        pat = re.compile(LEVEL_PATTERNS[level], re.IGNORECASE)
        pool = sorted({t for t in all_titles if pat.search(t)}, key=str.lower)
        if len(pool) >= MIN_TARGET_TITLES:
            return pool
    return neutral

def skills_text(skill_rows, rng):
    ranked = [s for s, hot in skill_rows if hot] + [s for s, hot in skill_rows if not hot]
    ranked = list(dict.fromkeys(ranked))[:8]
    if not ranked:
        return 'general'  # what the Findrr runner sends when no skills exist
    k = rng.randint(1, min(MAX_SKILLS, len(ranked)))
    return ', '.join(ranked[:k])

def build_title_examples(onet, split_of, seed=42):
    """Return {split: [{'input_text', 'target_text', 'soc'}]} in the runner's prompt format."""
    rng = random.Random(seed)
    out = {'train': [], 'val': [], 'test': []}
    for soc, occ_title in sorted(onet['occupations'].items()):
        if soc not in split_of:
            continue
        # ',' separates titles in the model output, so titles containing commas
        # (e.g. 'Superintendent, Schools') are left out of inputs and targets.
        base = [occ_title] if ',' not in occ_title else []
        reported = [t for t in onet['reported'].get(soc, []) if ',' not in t]
        all_titles = [t for t in dict.fromkeys(base + reported + onet['job_titles'].get(soc, []))
                      if ',' not in t]
        neutral = sorted({t for t in base + reported if not ANY_LEVEL.search(t)}, key=str.lower)
        if len(neutral) < MIN_TARGET_TITLES + 1:
            continue
        anchors = base + [t for t in neutral if not base or norm(t) != norm(occ_title)]
        anchors = anchors[:MAX_ANCHORS_PER_OCCUPATION]
        for level in LEVELS:
            pool = level_pool(level, neutral, all_titles)
            for anchor in anchors:
                targets = [t for t in pool if norm(t) != norm(anchor)][:MAX_TARGET_TITLES]
                if len(targets) < MIN_TARGET_TITLES:
                    continue
                out[split_of[soc]].append({
                    'input_text': f'expand titles: {anchor} | skills: '
                                  f'{skills_text(onet["skills"].get(soc, []), rng)} | level: {level}',
                    'target_text': ', '.join(targets),
                    'soc': soc,
                })
    return out

def title_dictionary(onet):
    """Every known O*NET title, normalised: the 'approved occupational dictionary'."""
    titles = set()
    for soc, t in onet['occupations'].items():
        titles.add(norm(t))
    for table in (onet['reported'], onet['job_titles']):
        for lst in table.values():
            titles.update(norm(t) for t in lst)
    return titles

def parse_titles(text):
    return [t.strip() for t in text.split(',') if t.strip()]

def valid_title_ratio(predictions, dictionary):
    titles = [t for p in predictions for t in parse_titles(p)]
    return sum(norm(t) in dictionary for t in titles) / len(titles) if titles else 0.0

def repetition_rate(predictions):
    rates = []
    for p in predictions:
        ts = [norm(t) for t in parse_titles(p)]
        rates.append((len(ts) - len(set(ts))) / len(ts) if ts else 0.0)
    return sum(rates) / len(rates) if rates else 0.0

In [ ]:
onet = load_onet(ONET_DIR)
split_of = split_by_occupation(onet['occupations'], seed=SEED)
examples = build_title_examples(onet, split_of, seed=SEED)
DICTIONARY = title_dictionary(onet)
for name, rows in examples.items():
    print(f'{name}: {len(rows)} examples from {len({r["soc"] for r in rows})} occupations')
overlap = {r['soc'] for r in examples['train']} & {r['soc'] for r in examples['test']}
assert not overlap, 'occupation leakage between train and test'
print('Sample:', examples['train'][0])

In [ ]:
import numpy as np, torch
from datasets import Dataset
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)
set_seed(SEED)
BASE_MODEL = 'google/flan-t5-small'
MAX_INPUT_LEN, MAX_TARGET_LEN = 128, 64
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

def tokenize(batch):
    enc = tokenizer(batch['input_text'], max_length=MAX_INPUT_LEN, truncation=True)
    enc['labels'] = tokenizer(text_target=batch['target_text'], max_length=MAX_TARGET_LEN,
                              truncation=True)['input_ids']
    return enc

ds = {k: Dataset.from_list(v).map(tokenize, batched=True, remove_columns=['input_text', 'target_text', 'soc'])
      for k, v in examples.items()}

# FP16 is off: T5 checkpoints are known to overflow in fp16 and the T4 has no bf16.
# flan-t5-small trains comfortably in fp32 on a T4.
args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/ml001-checkpoints',
    learning_rate=3e-4, lr_scheduler_type='cosine', warmup_ratio=0.1,
    num_train_epochs=5, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='eval_loss', save_total_limit=1,
    fp16=False, logging_steps=50, seed=SEED, report_to='none',
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'],
                         data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
                         processing_class=tokenizer)
trainer.train()

## Evaluation on held-out occupations (same decoding settings as the Findrr runner)

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu

def generate(texts, batch_size=64):
    model.eval(); outs = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=48, num_beams=2, do_sample=False,
                                 repetition_penalty=1.2)
        outs += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return outs

test_inputs = [r['input_text'] for r in examples['test']]
test_targets = [r['target_text'] for r in examples['test']]
preds = generate(test_inputs)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
measured = {
    'rougeL': float(np.mean([scorer.score(t, p)['rougeL'].fmeasure for t, p in zip(test_targets, preds)])),
    'bleu4': float(sacrebleu.corpus_bleu(preds, [test_targets]).score / 100.0),
    'valid_title_ratio': valid_title_ratio(preds, DICTIONARY),
    'repetition_rate': repetition_rate(preds),
    'test_examples': float(len(preds)),
}
demo = generate(['expand titles: Software Developer | skills: Python, SQL, Git | level: ANY'])[0]
measured['ac_ml_001_1_distinct_titles'] = float(len({norm(t) for t in parse_titles(demo)}))
print('Software Developer ->', demo)
print(json.dumps(measured, indent=2))

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        while chunk := fh.read(65536):
            h.update(chunk)
    return h.hexdigest()

def build_manifest(export_dir, model_id, model_name, version, architecture,
                   evaluation_summary, acceptance, metadata) -> dict:
    """Manifest in the exact schema of findrr.ml.manifest.ModelManifest.

    ``files`` maps every exported file (except manifest.json) to its SHA-256.
    ``evaluation_summary`` holds measured floats only; nothing is defaulted.
    """
    export_dir = Path(export_dir)
    files = {
        p.name: sha256_file(p)
        for p in sorted(export_dir.iterdir())
        if p.is_file() and p.name != 'manifest.json'
    }
    for key, value in evaluation_summary.items():
        if not isinstance(value, (int, float)) or value != value:  # reject None/NaN
            raise ValueError(f'Metric {key!r} was not measured: {value!r}')
    return {
        'model_id': model_id,
        'model_name': model_name,
        'version': version,
        'architecture': architecture,
        'export_date': datetime.now(timezone.utc).isoformat(),
        'files': files,
        'evaluation_summary': {k: float(v) for k, v in evaluation_summary.items()},
        'metadata': {**metadata, 'acceptance': acceptance},
    }

def check_acceptance(measured: dict, criteria: dict) -> dict:
    """criteria: {metric: ('>=' | '<', threshold)} taken verbatim from the model spec."""
    results = {}
    for metric, (op, threshold) in criteria.items():
        value = measured[metric]
        ok = value >= threshold if op == '>=' else value < threshold
        results[metric] = {'value': float(value), 'op': op, 'threshold': threshold, 'passed': bool(ok)}
    return {'passed': all(r['passed'] for r in results.values()), 'criteria': results}

In [ ]:
# Thresholds copied verbatim from docs/ml/models/title-expander.md (approved).
ACCEPTANCE = check_acceptance(measured, {
    'rougeL': ('>=', 0.75),
    'valid_title_ratio': ('>=', 0.85),
    'repetition_rate': ('<', 0.05),
    'ac_ml_001_1_distinct_titles': ('>=', 3),
})

In [ ]:
# Acceptance gate. A model that fails the approved criteria is NOT exported unless you
# deliberately set ALLOW_EXPORT_BELOW_ACCEPTANCE = True. In that case the manifest records
# acceptance.passed = false and the Findrr app refuses to load the model.
ALLOW_EXPORT_BELOW_ACCEPTANCE = False
print(json.dumps(ACCEPTANCE, indent=2))
if not ACCEPTANCE['passed'] and not ALLOW_EXPORT_BELOW_ACCEPTANCE:
    raise RuntimeError(
        'ML-001 did not meet its acceptance criteria. Not exporting. '
        'Report the metrics above; do not change thresholds without approval.'
    )

In [ ]:
import shutil, tarfile
EXPORT_DIR = Path('/kaggle/working/title-expander-v2.0.0')
shutil.rmtree(EXPORT_DIR, ignore_errors=True); EXPORT_DIR.mkdir(parents=True)
trainer.model.save_pretrained(EXPORT_DIR, safe_serialization=True)
tokenizer.save_pretrained(EXPORT_DIR)
(EXPORT_DIR / 'preprocess_config.json').write_text(json.dumps({
    'max_input_length': MAX_INPUT_LEN, 'max_target_length': MAX_TARGET_LEN,
    'input_template': 'expand titles: {title} | skills: {skills} | level: {level}',
    'levels': LEVELS, 'num_beams': 2, 'do_sample': False,
    'repetition_penalty': 1.2, 'max_new_tokens': 48,
}, indent=2))
import transformers
manifest = build_manifest(
    EXPORT_DIR, 'ML-001', 'title-expander', '2.0.0', 'google/flan-t5-small (seq2seq, safetensors)',
    evaluation_summary=measured, acceptance=ACCEPTANCE,
    metadata={'dataset': f'O*NET {ONET_VERSION} (CC BY 4.0)', 'dataset_sha256': ONET_SHA256,
              'split': 'by occupation, stratified by SOC major group (80/10/10)',
              'examples': {k: len(v) for k, v in examples.items()}, 'seed': SEED,
              'transformers': transformers.__version__, 'torch': torch.__version__},
)
(EXPORT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
BUNDLE = '/kaggle/working/title_expander_bundle.tar.gz'
with tarfile.open(BUNDLE, 'w:gz') as tar:
    for p in sorted(EXPORT_DIR.iterdir()):
        tar.add(p, arcname=p.name)
print(json.dumps(manifest, indent=2))

In [ ]:
# Re-verify the bundle exactly as the Findrr app will (SHA-256 of every file).
import tarfile, tempfile
with tempfile.TemporaryDirectory() as tmp:
    with tarfile.open('/kaggle/working/title_expander_bundle.tar.gz') as tar:
        tar.extractall(tmp)
    m = json.loads((Path(tmp) / 'manifest.json').read_text())
    for name, digest in m['files'].items():
        assert sha256_file(Path(tmp) / name) == digest, name
print('[OK] Bundle verified:', len(m['files']), 'files;', 'acceptance passed =', m['metadata']['acceptance']['passed'])